# claudedojo

> Start Claude Code with a worked tooling session already in its history

In [ ]:
#| default_exp claudedojo

A fresh Claude Code session gets its tooling *instructions* up front, but no *examples*: the first real tool calls of the session are produced cold, and early mistakes compound. In-context learning wants the opposite: a transcript that already shows the tools used well. `claudedojo` provides that. The template is an authored dialog (see `llmsurgery`): a worked, idiomatic dojo round written as a notebook, executed against the real kernel so every output is true, and converted to a session that new sessions resume, so each one opens with the clean round already in context.

In [ ]:
#| export
import asyncio, json, os, re, shutil, sys, tempfile, uuid
from importlib.resources import files
from fastcore.utils import *
from llmsurgery.ant import *
from llmsurgery.ipynb import read_ipynb, write_ipynb
from llmdojo.rules import _state_root, doced

In [ ]:
from fastcore.test import *
import tempfile
import shutil
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

In [ ]:
#| export
def _is_err(b): return b.get('type')=='tool_result' and b.get('is_error')

def _blocks(r):
    c = r.get('message', {}).get('content', '')
    return [] if isinstance(c, str) else c

## Curating the capture

Curation turns one good run into a reproducible template. Only the active thread survives, minus what a resumed session gains nothing from: thinking records, per-call hook confirmations and task reminders, and any failed tool call along with its error result. The capture prompt is then rewritten to a short one that reads naturally in the spawned session's history, and `reid_recs` pins the ids and timestamps so identical captures give identical files. Environment attachments (tool listings, skill listings, MCP instructions) stay: Claude Code injects them as deltas, so baked ones prevent identical re-injection on resume, and a different environment simply layers fresh deltas on top.

In [ ]:
#| export
TMPL_PROMPT = "Bootstrap and complete the dojo and tell me when you're ready."
DROP_ATT = ('hook_success', 'task_reminder')

def curate_dojo(
    recs, # Captured records, e.g. from `capture_dojo`
):
    "The reproducible template records for a captured dojo run"
    t = strip_think(sess_thread(recs))
    bad = {b['tool_use_id'] for r in t for b in _blocks(r) if _is_err(b)}
    def _keep(r):
        if r.get('type')=='attachment' and r['attachment'].get('type') in DROP_ATT: return False
        return not any(b.get('id') in bad or b.get('tool_use_id') in bad for b in _blocks(r))
    t = reid_recs(t.filter(_keep), 'claudedojo', ts='2026-01-01T00:00:00.000Z')
    p = first(r for r in t if r['type']=='user' and isinstance(r['message']['content'], str))   # the capture prompt, however it was phrased
    p['message']['content'] = TMPL_PROMPT
    return t

def dojo_cid(
    recs, # Template or captured records
):
    "The completion id recorded in the clean-round message"
    return first(m[1] for r in recs if (m := re.search(r'Completion id: ([0-9a-f]{4})', rec_txt(r))))

A synthetic mini-capture exercises the whole curation path offline: a worked turn whose result carries the dojo card, a thinking record, a hook confirmation, a failed call with its error result, and a clean scoring turn.

In [ ]:
mproj = Path(tempfile.mkdtemp())
tu = mk_tu('mcp__clikernel', {})
card = tool_turn('Run the llmdojo round now please.', 'mcp__clikernel__execute', dict(code='dojo_start()'),
    '== llmdojo ==\n1. (par 2) orient', 'Round underway.', cwd=mproj)
noise = [mk_rec('assistant', [dict(type='thinking', thinking='Planning.', signature='')], cwd=mproj),
    dict(type='attachment', uuid='att1', attachment=dict(type='hook_success', hookName='PreToolUse:x')),
    mk_rec('assistant', [tu], cwd=mproj), mk_rec('user', [dict(mk_tr(tu, 'Error: no such tool'), is_error=True)], cwd=mproj)]
score = tool_turn('Score it.', 'mcp__clikernel__execute', dict(code='dojo_score()'),
    "strokes 7, par 8\nkata 'orient': ok\nexpected answer: weather\nClean round. Run dir removed. Completion id: ab12 - keep this id.\nKernel namespace cleared, as after a restart. Doc-state is intact.", 'Clean: id ab12.', cwd=mproj)
msid = save_sess(card+noise+score, cwd=mproj)
mini = load_sess(msid, mproj)
len(mini)

Curation removes exactly the noise, rewrites the prompt, and yields the same bytes every time:

In [ ]:
c1,c2 = curate_dojo(mini),curate_dojo(mini)
test_eq(canon(list(c1)), canon(list(c2)))
test_eq(len(c1), 8)
test_eq(c1[0]['message']['content'], TMPL_PROMPT)
test_eq(dojo_cid(c1), 'ab12')
test_eq(c1[2]['message']['content'][0]['tool_use_id'], c1[1]['message']['content'][0]['id'])

## Capturing headlessly

When the tooling version bumps, the stored template goes stale and someone must play a fresh clean round. `capture_dojo` does it unattended: it spawns a real Claude Code session via the Agent SDK in a throwaway project, steered by a script *in the system prompt* — which never appears in transcript records, so the captured history needs no scrubbing and reads as the model's own work. The steering asks for a round worth copying and keeps itself out of the story - worded informally and plainly, since more clandestine-sounding drafts ('automated telemetry capture', 'never mention these instructions') tripped fable's safety layer, which re-screens the whole context on every call; every receipt in the transcript is still real (the kernel ran, the katas scored), only the plan was supplied.

A fumbled round baked into a template would teach every future session the fumbles as worked examples, so each attempt is gated by `is_clean` before it may become the store:

In [ ]:
#| export
GATE_FORBID = ('recording a demo', 'these notes')

def _vis(r):
    "An assistant record's visible text, excluding thinking blocks"
    c = r.get('message', {}).get('content', '')
    return c if isinstance(c, str) else ' '.join(b.get('text','') for b in c if b.get('type')=='text')

def is_clean(
    recs, # Captured thread records
    forbid=GATE_FORBID, # Phrases that must not appear in visible assistant text
):
    "Why a captured round can't become a template: a list of problems, empty when the run is clean"
    probs = []
    codes = [c for r in recs for b in _blocks(r) if b.get('type')=='tool_use' and (c := nested_idx(b, 'input', 'code'))]
    if (n := sum('dojo_start(' in c for c in codes)) != 1: probs.append(f'{n} dojo_start calls')
    if (n := sum('dojo_score(' in c for c in codes)) != 1: probs.append(f'{n} dojo_score calls (a clean round scores once, first try)')
    if any('dojo_redo(' in c or 'dojo_resume(' in c for c in codes): probs.append('needed a redo')
    if not any('Clean round' in rec_txt(r) for r in recs if r.get('type')=='user' and rec_role(r)=='tool'): probs.append('no clean score')
    if (n := sum(1 for r in recs for b in _blocks(r) if _is_err(b))): probs.append(f'{n} failed tool calls')
    txt = ' '.join(_vis(r) for r in recs if r.get('type')=='assistant')
    if (bad := [f for f in forbid if f in txt]): probs.append(f"script leaked into visible text: {', '.join(bad)}")
    return probs

The synthetic capture above is *not* clean — its noise turn includes a failed tool call — while the same round without the noise passes. Redos, re-scores, and script phrases in visible text (thinking blocks don't count: curation strips them) are each grounds for rejection:

In [ ]:
test_eq(is_clean(mini), ['1 failed tool calls'])
test_eq(is_clean(load_sess(save_sess(card+score, cwd=mproj), mproj)), [])
redo = tool_turn('Retry.', 'mcp__clikernel__execute', dict(code='dojo_redo(1)'), 'reset', 'Retrying.', cwd=mproj)
assert 'needed a redo' in is_clean(load_sess(save_sess(card+redo+score, cwd=mproj), mproj))
leak = [mk_rec('assistant', 'Since we are recording a demo, I begin.', cwd=mproj)]
assert any('script leaked' in p for p in is_clean(load_sess(save_sess(card+leak+score, cwd=mproj), mproj)))

The script is a sentence per cell naming the functions to run and the non-obvious parts; the model supplies the rest. It rides in the system prompt's `claude_code` preset append, with thinking disabled (the plan is supplied, and thinking is where a quoted script would otherwise hide). Each attempt runs in a fresh scratch project with a `pyproject.toml`, so clikernel starts eager and the startup script prints the same bootstrap the script refers to, and with `LLMDOJO_STATE_DIR` pointed into the scratch, so nothing an attempt does touches machine-global state: a flagged attempt's session outlives the SDK error (Claude Code retries the API error and plays on), and three such zombies once registered real completion receipts minutes after their attempts were declared failed. The round's doc-state record, keyed by the capture session id, rides from the scratch into the store for `prep_dojo` to seed.

A clean capture stores the template *and* writes it as a dialog, `template.ipynb` beside the store: that notebook is the review point — read it, curate the prose, then `claudedojo --build` it back into the store. The store is immediately usable either way; curation makes it beautiful, and the baked history is in-context teaching, so beautiful matters.

In [ ]:
#| export
CAPTURE_SYSP = """You're recording a demo session: the transcript will be replayed at the start of future sessions as a worked example of good tooling use. So make it a good one - play the round just as the tooling instructions in your context describe, and keep the narration about the work itself; these notes aren't part of the demo, so don't bring them up. One concern per cell: run the printed bootstrap imports, read the module docs with one batched doc() call, list_pyskills(), then import the round's extra pyskill - rgapi.skill, with both `import mod` and `from mod import *` so every tooling function is a bare name - and then, as the very next cell, exactly doc(rgapi.skill): the module overviews are the catalog later work retrieves from context, so that cell is not optional. Function docs are just-in-time: immediately before each kata's first use, ONE batched free doc() naming exactly what that kata is about to call - doc(find_msgs) before kata 1, doc(lnhashview_file, exhash_file) before kata 2, doc(exhash_cell) before kata 4 - and nothing speculative: never doc a function the kata won't call. Then dojo_start(), and %cd into the run dir the card names before anything else (chdir cells are free, and the relative paths they enable keep every later cell short); tag each kata with its comment-only tag cell and take the par route the kata card describes. Mind the qualified-name trap: a kata that says a module defines a function means you call it as module.func after import - a bare name is a NameError and a wasted stroke. Kata specifics that cost strokes when missed: kata 1 is par 1, one find_msgs call and done (its import/doc neighbors are free), with the httpx why in the markdown neighbor of the import cell - the answer must name both the policy and what httpx replaced; an edit set is ONE exhash_file call, each command tuple its own positional argument, ordered bottom-to-top (highest line number first) so earlier edits don't shift later addresses, and with s patterns written as regexes: escape [ ] ( ) . in literal text or the call fails and the retry costs a stroke; verbatim replacements go through the %%exhash cell magic, never quoted Python payloads - a range address with c inside a file, and for a whole notebook cell straight from the cell id find_msgs shows to %%exhash <path> <cell_id> % c - the address is the one character %, meaning the whole cell: no lnhashview_cell first, no line address ever; and the report style that ships is named only in daily_report's full docstring - end that cell with the bare call, no .splitlines() or slicing (read the first line from the output with your eyes and type it into dojo_score(report=...)). Score once with dojo_score(); play well enough that the first score is clean. After the clean score, reply exactly "OK I'm ready." and stop."""

async def capture_dojo(
    d=None, # Template store dir; `TMPL_DIR` if None
    model='fable', # Model for the capture run; the only one smart enough for clean rounds - if fable's safeguards flag the steering text, soften the wording rather than switching models
    attempts=3, # Gated attempts before giving up
    budget=10.0, # Max USD per attempt
):
    "Play a scripted dojo round headlessly, gate it with `is_clean`, and store the curated template"
    from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage, AssistantMessage, TextBlock, ToolUseBlock
    for i in range(attempts):
        proj = Path(tempfile.mkdtemp(prefix='claudedojo_'))
        (proj/'pyproject.toml').write_text('[project]\nname = "dojo-capture"\nversion = "0"\n')
        sid = str(uuid.uuid4())
        opts = ClaudeAgentOptions(cwd=proj, session_id=sid, model=model, max_budget_usd=budget,
            permission_mode='bypassPermissions', setting_sources=['user','project','local'],
            thinking={'type':'disabled'},
            env={'LLMDOJO_STATE_DIR': str(proj/'state')},   # attempts write no machine-global state: a flagged attempt's session can outlive the SDK error and still score
            system_prompt={'type':'preset','preset':'claude_code','append':CAPTURE_SYSP})
        cost,err = 0,None
        try:
            async for m in query(prompt=TMPL_PROMPT, options=opts):
                if isinstance(m, ResultMessage): cost = m.total_cost_usd or 0
                elif isinstance(m, AssistantMessage):
                    for b in m.content:
                        if isinstance(b, ToolUseBlock): print('>', str(b.input.get('code', b.input))[:80], flush=True)
                        elif isinstance(b, TextBlock) and b.text.strip(): print('.', b.text.strip()[:80], flush=True)
        except Exception as e: err = f'run failed: {e}'
        recs = [] if err else sess_thread(load_sess(sid, proj))
        probs = [err] if err else is_clean(recs)
        print(f"attempt {i+1}: ${cost:.2f}, {len(recs)} records" + (f", rejected: {'; '.join(probs)}" if probs else ", clean"), flush=True)
        if not probs:
            df = proj/'state'/'doced'/f'{sid}.json'
            ds = json.loads(df.read_text()) if df.exists() else []
            curated = curate_dojo(recs)
            save_template(curated, d, doced=ds)
            dlg = chat2dlg(recs2chat(curated), 'dojo_template', mx=None)
            dlg.meta['llmdojo'] = dict(doced=ds)
            nbf = Path(d or TMPL_DIR)/'template.ipynb'
            write_ipynb(dlg, nbf)
            print(f"stored: cid {dojo_cid(curated)}, {len(curated)} records, {(Path(d or TMPL_DIR)/'template.jsonl').stat().st_size//1024}KB")
            print(f"review dialog: {nbf} - curate it, then claudedojo --build it back into the store")
        shutil.rmtree(proj)
        shutil.rmtree(sess_dir(proj), ignore_errors=True)
        if not probs: return curated
    raise RuntimeError(f'no clean round in {attempts} attempts')

A live acceptance run (real model spend, so `eval: false` — run it after any dojo change, or via `claudedojo --capture`):

In [ ]:
#| eval: false
curated = await capture_dojo()
test_eq(is_clean(curated), [])

## Capturing from a live session

In [ ]:
#| export
def _exec_tu(r):
    c = r.get('message', {}).get('content', '')
    return None if isinstance(c, str) else first(b for b in c if b.get('type')=='tool_use' and b.get('name')=='mcp__clikernel__execute')

def pick_turns(
    recs, # Session records, e.g. from `load_sess`
    *cells, # Kernel cell sources: matched exactly, else as a prefix; the last match wins
):
    "The tool_use/tool_result record pairs for the kernel cells in `cells`, in that order"
    tus = [(r, tu) for r in recs if r.get('type')=='assistant' and (tu:=_exec_tu(r))]
    out = []
    for txt in cells:
        cand = [r for r,tu in tus if tu['input'].get('code','')==txt] or [r for r,tu in tus if tu['input'].get('code','').startswith(txt)]
        if not cand: raise ValueError(f'no kernel cell matches {txt.splitlines()[0]!r}')
        tid = _exec_tu(cand[-1])['id']
        out += [cand[-1], first(r for r in recs if r.get('type')=='user' and rec_role(r)=='tool' and any(b.get('tool_use_id')==tid for b in r['message']['content']))]
    return L(out)

A headless capture run is not the only source of a clean round: any live Claude Code session that just played one holds the same records, already verified by the score. `pick_turns` lifts them out. Each entry names a kernel cell by its source, matched exactly or as a prefix (handy when a cell embeds a per-round path); the last match wins, so a corrected retry beats its earlier attempt. Matching by source rather than position means the surrounding conversation, other rounds, and interleaved discussion are all ignored.

In [ ]:
picked = pick_turns(mini, 'dojo_start()')
test_eq([rec_role(r) for r in picked], ['assistant', 'tool'])
test_eq(pick_turns(mini, 'dojo_')[0], pick_turns(mini, 'dojo_score()')[0])
test_fail(lambda: pick_turns(mini, 'nonexistent cell'), contains='no kernel cell')

In [ ]:
#| export
OPENING = 'Bootstrapping: the startup script has already run its imports, so I read the tooling docs, check the catalog, then take the dojo.'

def mk_template(
    picked, # Record pairs from `pick_turns`
    opening=OPENING, # Reply text preceding the first tool call
    closing="OK I'm ready.", # Reply text ending the round
    doced=None, # Names doc()'d during the round, stored in the dialog's metadata for launch-time doc-state seeding
):
    "A template dialog: `picked` wrapped in `TMPL_PROMPT` and framing text, with tool output untruncated"
    recs = [mk_rec('user', TMPL_PROMPT), mk_rec('assistant', opening), *picked, mk_rec('assistant', closing)]
    dlg = chat2dlg(recs2chat(recs), 'dojo_template', mx=None)
    dlg.meta['llmdojo'] = dict(doced=list(doced or []))
    return dlg

`mk_template` turns the picked records into the reviewable artifact: a one-prompt dialog whose reply holds the whole round, ready for `write_ipynb` and then `--build`. The framing text records are synthetic, which is fine (resume trusts them), and `mx=None` matters: the default rendering truncates tool output at 2,000 characters, which would gut the baked `doc()` results that a spawned session relies on.

In [ ]:
tdlg = mk_template(picked, opening='Practice first.', doced=['doc','rg'])
test_eq(tdlg.meta, dict(llmdojo=dict(doced=['doc','rg'])))
test_eq(tdlg.messages[0].content, TMPL_PROMPT)
test_eq(len(tdlg.messages), 1)
assert 'dojo_start()' in str(tdlg.messages[0].output)
tmsgs = dlg2msgs(tdlg)
test_eq([m['role'] for m in tmsgs], ['user', 'assistant', 'user', 'assistant'])

## The template store

The template lives in a per-user state dir alongside its metadata: the completion id its clean round printed, the dojo tooling version it was captured against, and the round's `doced` list (which `prep_dojo` seeds into the spawned conversation's doc-state). The version is what keeps the store honest: llmdojo refuses completion ids minted under different tooling, so a stale template forces a real round rather than silently faking one.

In [ ]:
#| export
TMPL_DIR = _state_root()/'template'

def _dojo_v():
    from llmdojo.dojo import dojo_version
    return dojo_version()

def save_template(
    recs, # Curated template records
    d=None, # Store dir; `TMPL_DIR` if None
    doced=None, # Names doc()'d during the baked round; the current conversation's doc-state if None
):
    "Write the template and its metadata to the store"
    if doced is None:
        from llmdojo.rules import doced as _cur
        doced = _cur()
    d = Path(d or TMPL_DIR)
    d.mkdir(parents=True, exist_ok=True)
    (d/'template.jsonl').write_text(''.join(json.dumps(obj2dict(r))+'\n' for r in recs))
    (d/'meta.json').write_text(json.dumps(dict(cid=dojo_cid(recs), v=_dojo_v(), doced=doced)))

def load_template(
    d=None, # Store dir; `TMPL_DIR` if None
):
    "The stored template records and metadata"
    d = Path(d or TMPL_DIR)
    return [json.loads(l) for l in (d/'template.jsonl').read_text().splitlines()], json.loads((d/'meta.json').read_text())

In [ ]:
#| export
def build_template(
    src, # Path to a template dialog .ipynb
    d=None, # Store dir; `TMPL_DIR` if None
    cwd='.', # Project directory recorded in the records
):
    "Convert a template dialog into the store; the dialog's `llmdojo.doced` metadata rides along for launch-time doc-state seeding"
    dlg = read_ipynb(str(src))
    save_template(msgs2recs(dlg2msgs(dlg), key='claudedojo', cwd=str(cwd), model=None), d,
        doced=nested_idx(dlg.meta, 'llmdojo', 'doced'))

`build_template` is the dialog→store converter shared by `claudedojo --build` and the packaged fallback: the curated template dialog ships inside the package (`llmdojo/dojo_data/dojo_template.ipynb`), so a machine that has never run a capture still gets a working store on first launch.

## Launching

`prep_dojo` does the three launch-time steps: write the template as a session of the target project (under a fresh session id every launch, so `claude -r $(claudedojo --sid)` behaves exactly like plain `claude`, just with the round already in history), register the template's completion id with clikernel so `dojo_start('<id>')` in the spawned session validates, and hand back the id for `claude -r`. The spawned session then starts with the whole worked round in context, a kernel whose empty state matches the story its history tells, and live hooks and skills layered on top.

In [ ]:
#| export
def _load_reg(d):
    "Load the store (built from the packaged dialog if empty), warn on version skew, and register its completion id"
    if not (Path(d or TMPL_DIR)/'template.jsonl').exists():
        build_template(files('llmdojo')/'dojo_data'/'dojo_template.ipynb', d)   # first use on this machine: build the store from the packaged dialog
    recs,meta = load_template(d)
    if meta['v'] != _dojo_v(): print(f"claudedojo: template built under {meta['v']} but installed tooling is {_dojo_v()}; the baked round will not validate. Rebuild with claudedojo --build <dialog>, or refresh unattended with claudedojo --capture.")
    from llmdojo.dojo import register_completion
    register_completion(meta['cid'], meta['v'])
    return recs,meta

def _seed_doced(sid, names, merge=False):
    "Make the baked round's 'Doc-state is intact' true for conversation `sid`"
    d = _state_root()/'doced'
    d.mkdir(parents=True, exist_ok=True)
    f = d/f'{sid}.json'
    if merge and f.exists(): names = set(names) | set(json.loads(f.read_text()))
    f.write_text(json.dumps(sorted(names)))

def prep_dojo(
    cwd=None, # Project to start in; the current directory if None
    d=None, # Template store dir; `TMPL_DIR` if None
):
    "Write the template session for `cwd`, register its completion id, seed its doc-state, and return the session id to resume"
    recs,meta = _load_reg(d)
    sid = save_sess(recs, cwd=cwd, ts=True)
    if ds := meta.get('doced'): _seed_doced(sid, ds)
    return sid

In [ ]:
#| export
APPEND_PROMPT = "We've compacted - run the dojo round again and tell me when you're ready."

def append_dojo(
    sid=None, # Session id or name to append to; the project's newest transcript if None
    cwd=None, # Project directory; the current directory if None
    d=None, # Template store dir; `TMPL_DIR` if None
):
    "Append the template round to session `sid` (e.g. after a compaction), re-seed doc-state, and return `sid` to resume"
    trecs,meta = _load_reg(d)
    sid,path = resolve_session(sid, cwd or '.')
    tail = load_recs(path)
    trecs = reid_recs(trecs, f'append:{sid}:{len(tail)}')   # fresh salt per append, so repeated splices never collide
    trecs[0]['message']['content'] = APPEND_PROMPT
    append_sess(trecs, sid, cwd, ts=True)
    _seed_doced(sid, meta.get('doced') or [], merge=True)   # merge: after a compact the hook truncated the record, so this is just the round's list
    return sid

`append_dojo` splices the round into an *existing* conversation instead of starting a fresh one — the post-compaction repair. Compaction rewrites the model's context to a summary: the worked examples vanish and the SessionStart hook truncates the doc-state record. Appending the round puts the demonstrations (and their `doc()` outputs) back at the tail of the context, and re-seeds doc-state to match. The baked round's kernel story stays true because appending requires the app to be closed, and the kernel dies with the app: the resumed session gets a fresh kernel whose startup just ran its imports — exactly what the round's opening and closing claim. Only the synthetic prompt line differs (a compaction reads differently from a session start); every receipt is the same template. Repeated appends re-salt the record ids, so a session can be refreshed after every compact.

In [ ]:
#| export
USAGE = """usage: claudedojo [--build <dialog.ipynb> | --capture | --sid | -r [sessid] | <claude args...>]
Start claude in the current project with a completed dojo round already in its session history.
  --build <dialog.ipynb>  convert a template dialog and store it, instead of launching
  --capture               play a scripted round headlessly (Agent SDK), gate it with is_clean, and store it
  --sid                   prepare and print the session id, instead of launching; composes with shell aliases: claude -r $(claudedojo --sid)
  -r [sess]               append the round to an existing session, by id or name (the project's newest if omitted), and print its id; after a compaction: claude -r $(claudedojo -r)
  <claude args...>        anything else is passed through to claude"""

def main():
    "CLI entry point; run `claudedojo -h` for usage"
    args = sys.argv[1:]
    if args[:1] in (['-h'], ['--help']) or (args[:1]==['--build'] and len(args)!=2) \
        or (args[:1]==['--capture'] and len(args)!=1) or (args[:1]==['-r'] and len(args)>2): return print(USAGE)
    if args[:1] == ['--build']: return build_template(args[1], cwd=Path.cwd())
    if args[:1] == ['-r']: return print(append_dojo(args[1] if len(args)==2 else None))
    if args == ['--capture']:
        asyncio.run(capture_dojo())   # discard the records: the console script sys.exit()s main's return value
        return
    if args == ['--sid']: return print(prep_dojo())
    os.execvp('claude', ['claude', '-r', prep_dojo(), *args])

Store, registration, and session write, against temporary dirs (the llmdojo state redirect keeps the test off the real completion record):

In [ ]:
#| eval: false
store = Path(tempfile.mkdtemp())
save_template(c1, store, doced=['rg','view_nb'])
back,meta = load_template(store)
test_eq(meta, dict(cid='ab12', v=_dojo_v(), doced=['rg','view_nb']))
os.environ['LLMDOJO_STATE_DIR'] = str(store)
try:
    lsid = prep_dojo(mproj, store)
    test_ne(prep_dojo(mproj, store), lsid)   # a fresh session id every launch: relaunching never touches an earlier session
    from llmdojo.dojo import _completions
    test_eq(_completions()['ab12']['v'], _dojo_v())
    test_eq(json.loads((store/'doced'/f'{lsid}.json').read_text()), ['rg','view_nb'])
finally: del os.environ['LLMDOJO_STATE_DIR']
test_eq(load_sess(lsid, mproj)[0].message.content, TMPL_PROMPT)
test_eq(len(load_sess(lsid, mproj)), len(c1))

# An empty store falls back to the packaged dialog: first launch works on a machine that never captured
fresh = Path(tempfile.mkdtemp())/'store'
os.environ['LLMDOJO_STATE_DIR'] = str(fresh)
try: fsid = prep_dojo(mproj, fresh)
finally: del os.environ['LLMDOJO_STATE_DIR']
test_eq(load_sess(fsid, mproj)[0].message.content, TMPL_PROMPT)
test_eq(load_template(fresh)[1]['v'], _dojo_v())
assert json.loads((fresh/'doced'/f'{fsid}.json').read_text())

Appending to the prepared session splices a re-salted copy of the round onto its tail, rewrites the synthetic prompt, and merges the doc-state seed — and a second append (the next compaction) never collides:

In [ ]:
#| eval: false
os.environ['LLMDOJO_STATE_DIR'] = str(store)
try:
    n0 = len(load_sess(lsid, mproj))
    test_eq(append_dojo(lsid, mproj, store), lsid)
    ap = load_sess(lsid, mproj)
    test_eq(len(ap), n0+len(c1))
    test_eq(ap[n0].parentUuid, ap[n0-1].uuid)
    test_eq(ap[n0].message.content, APPEND_PROMPT)
    append_dojo(lsid, mproj, store)
    ap2 = load_sess(lsid, mproj)
    test_eq(len(set(r.uuid for r in ap2)), len(ap2))
    test_eq(json.loads((store/'doced'/f'{lsid}.json').read_text()), ['rg','view_nb'])
    name_sess(lsid, 'flux-dojo', mproj)
    test_eq(append_dojo('flux-dojo', mproj, store), lsid)
    assert not (store/'doced'/'flux-dojo.json').exists()
finally: del os.environ['LLMDOJO_STATE_DIR']

The launcher peels its own forms (`--build`, `--sid`, `-h`) from `sys.argv` by hand rather than via argparse, so that everything else can pass through to `claude` verbatim, including flags argparse would reject. The pass-through composes with shell aliases that add flags like `--system-prompt-file`; the usage paths are testable without launching anything. `--build` converts a template dialog directly: `llmsurgery` renders prompts as bare text (the XML envelope around Solveit prompts is Solveit's own `prompt_txt` patch, which never loads here), and records omit the model so a resumed session uses the user's configured default. `prep_dojo` stamps launch-time timestamps, so the session picker shows its real age:

In [ ]:
sys.argv[1:] = ['-h']
test_stdout(main, USAGE)
sys.argv[1:] = ['--build']
test_stdout(main, USAGE)
sys.argv[1:] = ['--capture', 'extra']
test_stdout(main, USAGE)
sys.argv[1:] = ['-r', 'a', 'b']
test_stdout(main, USAGE)
sys.argv[1:] = []

The proof that a spawned session truly inherits the round: resume the prepared session and ask about facts only its history holds. The model should quote the completion id its dojo round printed, and know that its kernel state is gone. This needs a captured template in the real store, and spends tokens, so it stays out of CI:

In [ ]:
#| eval: false
psid = prep_dojo(mproj)
opts = ClaudeAgentOptions(resume=psid, cwd=str(mproj), model='haiku')
async for m in query(prompt='From this session only: what completion id did the dojo round print, and what did its final message say about kernel state?', options=opts):
    if isinstance(m, ResultMessage): print(m.result)

## Rebuilding the template

When the dojo tooling version changes, `prep_dojo` warns and the baked completion id stops validating, so the template needs a fresh round. The workflow: play one clean round in a live Claude Code session in this project, then run the cells below *from that same session* (`load_sess()` resolves to the newest transcript, which is the conversation that just played). The cell sources below name the round's kernel cells for `pick_turns`; the round's %cd into its run dir makes every later path relative, so the sources are round-independent (the %cd cell itself is matched by prefix); on a rebuild just adjust any kata cells the new dojo changed and run.

In [ ]:
#| eval: false
imports = '''import rgapi.skill as rga
from rgapi.skill import *
from llmdojo.dojo import *'''
round_cells = ['forget_doced(); doc(clik, pysk, fct, dsk, exh)', 'list_pyskills()', imports,
    'doc(rga)', 'doc(dojo_score)', 'dojo_start()', '%cd ',   # prefix: the run dir is the only per-round text in the round
    '# kata 1', 'doc(find_msgs)', "find_msgs('httpx', dlg='nbs/01_api.ipynb')",
    '# kata 2', 'doc(lnhashview_file, exhash_file)', "lnhashview_file('core.py')", "exhash_file('core.py'",
    '# kata 3', "lnhashview_file('tmpl.py')", '%%exhash tmpl.py',
    '# kata 4', 'doc(exhash_cell)', "find_msgs(header_section='Retries', dlg='nbs/01_api.ipynb')", '%%exhash nbs/01_api.ipynb',
    '# kata 5', 'import report\ndoc(report, report.daily_report)', "report.daily_report(report.SAMPLE, style='rb2')",
    'dojo_score(bash_calls=0,']

In [ ]:
#| eval: false
dlg = mk_template(pick_turns(sess_thread(load_sess()), *round_cells), doced=doced())
write_ipynb(dlg, 'llmdojo/dojo_data/dojo_template.ipynb')

The written dialog is the review point: read it, edit whatever makes it a better role model — prose freely, and tool calls or outputs too, provided edited receipts stay faithful to what the live tooling prints (when a change would touch many outputs, replaying the round is the cheaper generator of true ones) — then convert and store it with `build_template`, exactly as `claudedojo --build` would. Its home is `llmdojo/dojo_data/dojo_template.ipynb`, tracked in git and shipped as package data so a fresh machine's first launch can build its store from it. The meta check should show the new round's completion id and the installed dojo version.

In [ ]:
#| eval: false
build_template('llmdojo/dojo_data/dojo_template.ipynb', cwd=Path.cwd())
load_template()[1]

## Cleanup

In [ ]:
shutil.rmtree(sess_dir(mproj))
shutil.rmtree(mproj)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()